<a href="https://colab.research.google.com/github/dougyd92/ML-Foudations/blob/main/Notebooks/13_Neural_Networks_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 13: Introduction to Neural Networks

This notebook accompanies the Session 13 slides. We'll explore neural networks by connecting them to models you already know, building up from logistic regression to multi-layer networks, and implementing forward propagation from scratch.

**Key ideas:**
- A perceptron is logistic regression with different vocabulary
- Hidden layers create new feature representations
- Nonlinear activation functions are what make depth useful
- Forward propagation is just repeated matrix multiplies + activations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs, load_digits, load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All imports successful!")

---
# Helper functions

We'll reuse this function throughout the notebook to plot decision boundaries for different classifiers.

In [ ]:
def plot_decision_boundary(ax, model, X, y, title="", show_accuracy=True):
    """Plot the decision boundary of a trained classifier."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    cmap_light = ListedColormap(['#D6EAF8', '#FADBD8'])
    cmap_bold = ListedColormap(['steelblue', 'coral'])

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold,
               edgecolors='white', linewidth=0.5, s=40, alpha=0.8)
    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())

    if show_accuracy:
        acc = accuracy_score(y, model.predict(X))
        title_str = f"{title}\nAccuracy: {acc:.1%}"
    else:
        title_str = title

    ax.set_title(title_str, fontsize=13)
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')

print("Helper function defined.")

---
# Section 1: The perceptron is logistic regression

A perceptron with a sigmoid activation function and no hidden layers is mathematically identical to logistic regression. Let's verify this in code.

We'll generate a simple linearly separable dataset and train both models on it. If they're really equivalent, they should learn the same decision boundary.

In [ ]:
# Generate a simple 2D dataset
from sklearn.datasets import make_classification

X_simple, y_simple = make_classification(
    n_samples=200, n_features=2, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)

# Standardize features
scaler_simple = StandardScaler()
X_simple = scaler_simple.fit_transform(X_simple)

print(f"Dataset shape: {X_simple.shape}")
print(f"Class distribution: {np.bincount(y_simple)}")

In [ ]:
# Plot X_simple, y_simple
fig, ax = plt.subplots(figsize=(4, 4))
cmap = ListedColormap(['steelblue', 'coral'])
ax.scatter(X_simple[:, 0], X_simple[:, 1], c=y_simple, cmap=cmap)
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Simple dataset')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Now train logistic regression and a "perceptron" (MLPClassifier with no hidden layers). The key argument is `hidden_layer_sizes=()`, which means zero hidden layers: just input → activation → output.

In [ ]:
# Logistic regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_simple, y_simple)

# "Perceptron" = MLPClassifier with no hidden layers, logistic activation
# hidden_layer_sizes=() means: no hidden layers
perceptron = MLPClassifier(
    hidden_layer_sizes=(),    # no hidden layers
    activation='logistic',     # sigmoid
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
perceptron.fit(X_simple, y_simple)

print(f"Logistic Regression accuracy: {lr.score(X_simple, y_simple):.4f}")
print(f"Perceptron accuracy:          {perceptron.score(X_simple, y_simple):.4f}")

In [ ]:
# Compare decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_decision_boundary(axes[0], lr, X_simple, y_simple,
                       title="Logistic Regression")
plot_decision_boundary(axes[1], perceptron, X_simple, y_simple,
                       title="Perceptron (0 hidden layers)")

plt.tight_layout()
plt.show()

The boundaries are essentially identical. Different vocabulary, same math.

In [ ]:
# Compare the learned weights
print("Logistic Regression coefficients:")
print(f"  Weights: {lr.coef_[0]}")
print(f"  Bias:    {lr.intercept_[0]:.4f}")
print()
print("Perceptron coefficients:")
print(f"  Weights: {perceptron.coefs_[0].flatten()}")
print(f"  Bias:    {perceptron.intercepts_[0][0]:.4f}")
print()
print("The values may differ slightly (different optimization paths),")
print("but the decision boundaries are the same.")

---
# Section 2: Model comparison on noisy XOR

XOR is the classic problem that a single perceptron (or logistic regression) can't solve. Let's generate a noisy XOR dataset and see how different models handle it.

We'll create four Gaussian clusters arranged at the XOR corners: (0,0) and (1,1) are class 0, while (0,1) and (1,0) are class 1. Adding noise makes it a realistic classification problem rather than just four clean points.

In [ ]:
# Generate noisy XOR dataset
centers = [[0, 0], [0, 1], [1, 0], [1, 1]]
X_xor, y_cluster = make_blobs(n_samples=400, centers=centers,
                               cluster_std=0.25, random_state=42)

# XOR labels: class 1 when inputs differ, class 0 when they match
y_xor = np.array([0, 1, 1, 0])[y_cluster]

print(f"Dataset shape: {X_xor.shape}")
print(f"Class distribution: {np.bincount(y_xor)}")

In [ ]:
# Quick look at the data
fig, ax = plt.subplots(figsize=(7, 6))
cmap = ListedColormap(['steelblue', 'coral'])
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap=cmap,
           edgecolors='white', linewidth=0.5, s=40, alpha=0.8)
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Noisy XOR dataset')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Now let's train four models and compare their decision boundaries. All four use the same sklearn API: instantiate, fit, predict.

In [ ]:
# Train four models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Neural Network (1 hidden layer w/ 4 neurons)": MLPClassifier(
        hidden_layer_sizes=(4,),
        activation='relu',
        max_iter=5000,
        random_state=42
    )
}

for name, model in models.items():
    model.fit(X_xor, y_xor)
    acc = accuracy_score(y_xor, model.predict(X_xor))
    print(f"{name:25s} accuracy: {acc:.4f}")

In [ ]:
# Plot all four decision boundaries
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, (name, model) in zip(axes, models.items()):
    plot_decision_boundary(ax, model, X_xor, y_xor, title=name)

plt.suptitle("Model comparison on noisy XOR", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

Notice the differences:
- **Logistic regression** draws a single linear boundary and gets about 50% (essentially random guessing)
- **Decision tree** carves out rectangular regions with axis-aligned splits
- **KNN** follows the local neighborhood structure
- **Neural network** learns a smooth, nonlinear boundary

All four use the same `.fit()` / `.predict()` API. The only difference is the model's internal structure.

---
## Exercise 1: Experimenting with network architecture

### Task 1: Minimum architecture for XOR

What's the smallest neural network that can solve the noisy XOR problem? Try different `hidden_layer_sizes` and find the simplest architecture that achieves > 90% accuracy.

Hint: `hidden_layer_sizes=(2,)` means one hidden layer with 2 neurons. `(4, 2)` means two hidden layers, first with 4 neurons, second with 2.

In [ ]:
# Try different architectures and report their accuracy
# Architectures to try: (1,), (2,), (3,), (4,), (2, 2), (4, 2)

# Your code here

In [ ]:
#@title Click to reveal solution

architectures = [(1,), (2,), (3,), (4,), (8,), (2, 2), (4, 2), (16, 8)]

print("Architecture        Accuracy")
print("-" * 35)

for arch in architectures:
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=5000,
        random_state=42
    )
    mlp.fit(X_xor, y_xor)
    acc = accuracy_score(y_xor, mlp.predict(X_xor))
    print(f"{str(arch):20s} {acc:.4f}")

### Task 2: Visualize architecture effects

Pick three architectures (one too small, one medium, one large) and plot their decision boundaries side by side.

In [ ]:
# Plot decision boundaries for 3 different architectures

# Your code here

In [ ]:
#@title Click to reveal solution

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
arch_list = [(1,), (8,), (16, 8)]
labels = ["Too small: (1,)", "Moderate: (8,)", "Large: (16, 8)"]

for ax, arch, label in zip(axes, arch_list, labels):
    mlp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=5000,
        random_state=42
    )
    mlp.fit(X_xor, y_xor)
    plot_decision_boundary(ax, mlp, X_xor, y_xor, title=label)

plt.suptitle("Effect of architecture on decision boundary", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
# Section 3: Activation functions

Activation functions introduce nonlinearity, which is what makes stacking layers useful. Let's visualize the three main activation functions and understand why they matter.

In [ ]:
# Define activation functions
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def tanh(z):
    return np.tanh(z)

def relu(z):
    return np.maximum(0, z)

z = np.linspace(-5, 5, 200)

In [ ]:
# Plot all three activation functions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sigmoid
axes[0].plot(z, sigmoid(z), color='steelblue', linewidth=2)
axes[0].axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
axes[0].axhline(y=0.5, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
axes[0].axhline(y=1, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linewidth=0.5, linestyle='-')
axes[0].set_title('Sigmoid: σ(z) = 1 / (1 + e⁻ᶻ)')
axes[0].set_xlabel('z')
axes[0].set_ylabel('σ(z)')
axes[0].set_ylim(-0.1, 1.1)
axes[0].text(2.5, 0.15, 'Range: (0, 1)', fontsize=11, color='steelblue')
axes[0].grid(True, alpha=0.3)

# Tanh
axes[1].plot(z, tanh(z), color='coral', linewidth=2)
axes[1].axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
axes[1].axhline(y=-1, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].axhline(y=1, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].axvline(x=0, color='gray', linewidth=0.5, linestyle='-')
axes[1].set_title('Tanh: tanh(z)')
axes[1].set_xlabel('z')
axes[1].set_ylabel('tanh(z)')
axes[1].set_ylim(-1.3, 1.3)
axes[1].text(2.0, -0.8, 'Range: (-1, 1)', fontsize=11, color='coral')
axes[1].grid(True, alpha=0.3)

# ReLU
axes[2].plot(z, relu(z), color='forestgreen', linewidth=2)
axes[2].axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
axes[2].axvline(x=0, color='gray', linewidth=0.5, linestyle='-')
axes[2].set_title('ReLU: max(0, z)')
axes[2].set_xlabel('z')
axes[2].set_ylabel('ReLU(z)')
axes[2].set_ylim(-1, 5.5)
axes[2].text(2.0, 1.0, 'Range: [0, ∞)', fontsize=11, color='forestgreen')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Why nonlinearity matters: the linear collapse

What happens if we stack two linear transformations without any activation function? Let's see.

In [ ]:
# Two separate linear transformations
W1 = np.array([[2, -1],
               [1,  3]])
b1 = np.array([1, -1])

W2 = np.array([[0.5, 2],
               [-1,  1]])
b2 = np.array([0, 1])

# Apply them sequentially (no activation)
x = np.array([0.8, -0.4])

z1 = W1 @ x + b1
z2 = W2 @ z1 + b2
print(f"Two layers applied sequentially: {z2}")

# Now combine into one equivalent transformation
W_combined = W2 @ W1
b_combined = W2 @ b1 + b2

z_single = W_combined @ x + b_combined
print(f"Single combined layer:            {z_single}")
print(f"Same result? {np.allclose(z2, z_single)}")
print()
print("Two linear layers without activation = one linear layer.")
print("The extra layer bought us nothing!")

Now with a nonlinear activation between the layers, the result is different: it can't be collapsed into a single linear transformation.

In [ ]:
# With ReLU activation between the layers
z1 = W1 @ x + b1
a1 = relu(z1)          # <-- nonlinear activation
z2 = W2 @ a1 + b2

print(f"With ReLU between layers: {z2}")
print(f"Same as single layer?    {np.allclose(z2, z_single)}")
print()
print("The activation function breaks the collapse.")
print("Now the two layers can represent something a single layer cannot.")

---
## Exercise 2: Implementing activation functions

### Task 1: Implement activation functions and their derivatives

Implement ReLU, sigmoid, and their derivatives. The derivatives will be useful in Session 14 when we cover backpropagation, but it's good to see them now.

Recall:
- Sigmoid derivative: σ'(z) = σ(z) · (1 - σ(z))
- ReLU derivative: 1 if z > 0, else 0

In [ ]:
# Implement the derivatives
# sigmoid is already defined above

def sigmoid_derivative(z):
    # Hint: use the sigmoid function you already have
    # Your code here
    pass

def relu_derivative(z):
    # Hint: returns 1 where z > 0, else 0
    # Your code here
    pass

In [ ]:
#@title Click to reveal solution

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu_derivative(z):
    return (z > 0).astype(float)

### Task 2: Plot the activation functions alongside their derivatives

In [ ]:
# Plot sigmoid and ReLU with their derivatives side by side
# Use z = np.linspace(-5, 5, 200)

# Your code here

In [ ]:
#@title Click to reveal solution

z = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid and its derivative
axes[0].plot(z, sigmoid(z), color='steelblue', linewidth=2, label='σ(z)')
axes[0].plot(z, sigmoid_derivative(z), color='coral', linewidth=2,
             linestyle='--', label="σ'(z)")
axes[0].set_title('Sigmoid and its derivative')
axes[0].set_xlabel('z')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='gray', linewidth=0.5)
axes[0].axvline(x=0, color='gray', linewidth=0.5)

# ReLU and its derivative
axes[1].plot(z, relu(z), color='forestgreen', linewidth=2, label='ReLU(z)')
axes[1].plot(z, relu_derivative(z), color='coral', linewidth=2,
             linestyle='--', label="ReLU'(z)")
axes[1].set_title('ReLU and its derivative')
axes[1].set_xlabel('z')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(-0.5, 5.5)
axes[1].axhline(y=0, color='gray', linewidth=0.5)
axes[1].axvline(x=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Notice: sigmoid's derivative peaks at 0.25 and gets tiny near the edges.")
print("This is the 'vanishing gradient' problem we'll discuss in Session 14.")
print("ReLU's derivative is either 0 or 1. No vanishing gradient for positive inputs.")

---
# Section 4: Forward propagation in NumPy

Now we'll implement forward propagation from scratch using only NumPy. This is the same computation we traced by hand on the slides, but now in code.

**Network architecture:**
- 2 inputs
- 1 hidden layer with 2 neurons (ReLU activation)
- 1 output neuron (sigmoid activation)

### Step 1: Define the network weights

These are the same weights from the slide example. In a real network, these would be learned during training. For now, we set them manually.

In [ ]:
# Hidden layer weights and biases (2 inputs -> 2 hidden neurons)
W1 = np.array([[0.5, -0.3],    # neuron 1 weights: w11=0.5, w12=-0.3
               [-0.4, 0.8]])   # neuron 2 weights: w21=-0.4, w22=0.8
b1 = np.array([0.1, 0.2])      # biases: b1=0.1, b2=0.2

# Output layer weights and biases (2 hidden neurons -> 1 output)
W2 = np.array([[0.6, -0.5]])   # output neuron weights
b2 = np.array([0.1])           # output bias

# Input
x = np.array([1, 0])

print("Network weights defined.")
print(f"Input: x = {x}")
print(f"Hidden layer: W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"Output layer: W2 shape = {W2.shape}, b2 shape = {b2.shape}")

### Step 2: Hidden layer computation

Compute the weighted sum for each hidden neuron, then apply ReLU.

In [ ]:
# Weighted sums for hidden layer
z1 = W1 @ x + b1
print(f"Hidden layer weighted sums: z1 = {z1}")
print(f"  Neuron 1: z = (0.5)(1) + (-0.3)(0) + 0.1 = {z1[0]}")
print(f"  Neuron 2: z = (-0.4)(1) + (0.8)(0) + 0.2 = {z1[1]}")

In [ ]:
# Apply ReLU activation
a1 = relu(z1)
print(f"Hidden layer activations: a1 = {a1}")
print(f"  Neuron 1: ReLU({z1[0]}) = max(0, {z1[0]}) = {a1[0]}")
print(f"  Neuron 2: ReLU({z1[1]}) = max(0, {z1[1]}) = {a1[1]}")
print()
print("Notice: neuron 2 has a negative weighted sum, so ReLU outputs 0.")
print("This neuron is 'dead' for this particular input.")

### Step 3: Output layer computation

Take the hidden layer's output, compute the weighted sum, and apply sigmoid.

In [ ]:
# Weighted sum for output layer
z2 = W2 @ a1 + b2
print(f"Output weighted sum: z2 = {z2[0]:.4f}")

# Apply sigmoid activation
y_hat = sigmoid(z2)
print(f"Output (sigmoid): ŷ = σ({z2[0]:.4f}) = {y_hat[0]:.4f}")
print()

# Classification
predicted_class = 1 if y_hat[0] > 0.5 else 0
print(f"Prediction: ŷ = {y_hat[0]:.4f} > 0.5? {'Yes' if predicted_class == 1 else 'No'}")
print(f"Predicted class: {predicted_class}")

### Putting it all together: a reusable forward pass function

In [ ]:
def forward_pass(x, W1, b1, W2, b2):
    """
    Forward propagation through a 2-layer network.

    Layer 1: ReLU activation
    Layer 2: Sigmoid activation

    Returns all intermediate values for inspection.
    """
    # Hidden layer
    z1 = W1 @ x + b1
    a1 = relu(z1)

    # Output layer
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)

    return z1, a1, z2, a2

# Test with our example
z1, a1, z2, a2 = forward_pass(x, W1, b1, W2, b2)
print(f"Input:             x  = {x}")
print(f"Hidden weighted:   z1 = {z1}")
print(f"Hidden activated:  a1 = {a1}")
print(f"Output weighted:   z2 = {z2}")
print(f"Output activated:  a2 = {a2}")

### Processing multiple inputs

In practice, we pass many inputs at once using matrix multiplication. Each row is one input sample.

In [ ]:
# Multiple inputs (batch of 4)
X_batch = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

print("Forward pass for each XOR input:")
print("-" * 50)
for i, x_i in enumerate(X_batch):
    _, _, _, output = forward_pass(x_i, W1, b1, W2, b2)
    pred = 1 if output[0] > 0.5 else 0
    print(f"  Input: {x_i}  →  Output: {output[0]:.4f}  →  Class: {pred}")

print()
print("With random weights, the predictions are meaningless.")
print("The network needs training to learn useful weights.")
print("That's Session 14!")

---
## Exercise 3: Forward propagation on a larger network

### Task 1: Implement forward propagation for a 3-layer network

Extend the forward pass to handle a network with:
- 3 inputs
- Hidden layer 1: 4 neurons, ReLU
- Hidden layer 2: 3 neurons, ReLU
- Output layer: 2 neurons, softmax (multi-class)

For softmax, use: `softmax(z) = exp(z) / sum(exp(z))`

In [ ]:
# Network weights (randomly initialized for this exercise)
np.random.seed(42)
W1_big = np.random.randn(4, 3) * 0.5   # 3 inputs -> 4 hidden
b1_big = np.zeros(4)
W2_big = np.random.randn(3, 4) * 0.5   # 4 hidden -> 3 hidden
b2_big = np.zeros(3)
W3_big = np.random.randn(2, 3) * 0.5   # 3 hidden -> 2 output
b3_big = np.zeros(2)

# Test input
x_test = np.array([0.5, -1.0, 2.0])

# Implement the forward pass through all three layers
# Use ReLU for hidden layers, softmax for output
# Hint: softmax(z) = np.exp(z) / np.sum(np.exp(z))

# Your code here

In [ ]:
#@title Click to reveal solution

def softmax(z):
    exp_z = np.exp(z - np.max(z))  # subtract max for numerical stability
    return exp_z / np.sum(exp_z)

# Layer 1
z1 = W1_big @ x_test + b1_big
a1 = relu(z1)
print(f"Layer 1 weighted sums: {z1}")
print(f"Layer 1 activations:   {a1}")
print()

# Layer 2
z2 = W2_big @ a1 + b2_big
a2 = relu(z2)
print(f"Layer 2 weighted sums: {z2}")
print(f"Layer 2 activations:   {a2}")
print()

# Output layer
z3 = W3_big @ a2 + b3_big
a3 = softmax(z3)
print(f"Output weighted sums:  {z3}")
print(f"Output (softmax):      {a3}")
print(f"Probabilities sum to:  {a3.sum():.4f}")
print(f"Predicted class:       {np.argmax(a3)}")

### Task 2: Count the parameters

How many trainable parameters (weights + biases) does this 3-layer network have? Calculate it by hand, then verify with code.

In [ ]:
# Count parameters for the 3-layer network
# Layer 1: (inputs x neurons) + neurons = ?
# Layer 2: ?
# Layer 3: ?
# Total: ?

# Your code here

In [ ]:
#@title Click to reveal solution

layer1_params = W1_big.size + b1_big.size
layer2_params = W2_big.size + b2_big.size
layer3_params = W3_big.size + b3_big.size
total = layer1_params + layer2_params + layer3_params

print(f"Layer 1: ({W1_big.shape[1]} x {W1_big.shape[0]}) + {b1_big.shape[0]} = {layer1_params}")
print(f"Layer 2: ({W2_big.shape[1]} x {W2_big.shape[0]}) + {b2_big.shape[0]} = {layer2_params}")
print(f"Layer 3: ({W3_big.shape[1]} x {W3_big.shape[0]}) + {b3_big.shape[0]} = {layer3_params}")
print(f"Total:   {total} parameters")

---
# Section 5: Comparing our forward pass to MLPClassifier

Let's verify that our manual implementation matches what sklearn does inside `.predict()`. We'll train an MLPClassifier, extract its learned weights, and run our forward pass function with those same weights.

In [ ]:
# Train a small MLPClassifier on the noisy XOR data
mlp = MLPClassifier(
    hidden_layer_sizes=(4,),
    activation='relu',
    solver='adam',
    max_iter=2000,
    random_state=42
)
mlp.fit(X_xor, y_xor)

# Extract the learned weights
W1_learned = mlp.coefs_[0].T     # sklearn stores as (input, output), we want (output, input)
b1_learned = mlp.intercepts_[0]
W2_learned = mlp.coefs_[1].T
b2_learned = mlp.intercepts_[1]

print(f"Hidden layer weights shape: {W1_learned.shape}")
print(f"Hidden layer biases shape:  {b1_learned.shape}")
print(f"Output layer weights shape: {W2_learned.shape}")
print(f"Output layer biases shape:  {b2_learned.shape}")

In [ ]:
# Compare our forward pass to sklearn's predictions
test_points = X_xor[:5]

print("Comparing manual forward pass to sklearn .predict_proba():")
print("-" * 60)
for i, x_i in enumerate(test_points):
    # Our forward pass
    z1 = W1_learned @ x_i + b1_learned
    a1 = relu(z1)
    z2 = W2_learned @ a1 + b2_learned
    our_output = sigmoid(z2)[0]

    # Sklearn's prediction
    sklearn_prob = mlp.predict_proba(x_i.reshape(1, -1))[0, 1]

    print(f"  Point {i}: ours = {our_output:.6f}, sklearn = {sklearn_prob:.6f}, "
          f"match = {np.isclose(our_output, sklearn_prob, atol=1e-4)}")

print()
print("The outputs match! Every call to .predict() does exactly")
print("the forward pass we just implemented by hand.")

---
## Exercise 4: Neural network on a real dataset

### Task 1: Train and evaluate on breast cancer data

Load the breast cancer dataset, split into train/test, scale the features, and train both a logistic regression and an MLPClassifier. Compare their performance.

Hint: use `StandardScaler`, fit on train only, transform both.

In [ ]:
# Load the breast cancer dataset
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target

print(f"Dataset: {X_bc.shape[0]} samples, {X_bc.shape[1]} features")
print(f"Classes: {data.target_names}")
print(f"Class distribution: {np.bincount(y_bc)}")

# Split, scale, train logistic regression and MLPClassifier
# Compare accuracy on the test set

# Your code here

In [ ]:
#@title Click to reveal solution

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # transform, NOT fit_transform

# Logistic regression baseline
lr_bc = LogisticRegression(random_state=42, max_iter=1000)
lr_bc.fit(X_train_scaled, y_train)

# Neural network
mlp_bc = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    max_iter=1000,
    random_state=42
)
mlp_bc.fit(X_train_scaled, y_train)

print(f"Logistic Regression - Train: {lr_bc.score(X_train_scaled, y_train):.4f}, "
      f"Test: {lr_bc.score(X_test_scaled, y_test):.4f}")
print(f"Neural Network      - Train: {mlp_bc.score(X_train_scaled, y_train):.4f}, "
      f"Test: {mlp_bc.score(X_test_scaled, y_test):.4f}")

### Task 2: Experiment with architectures

Try at least three different architectures. Which works best on the test set? Does bigger always mean better?

In [ ]:
# Try different architectures and compare test accuracy
# Suggestions: (8,), (32, 16), (64, 32, 16), (128, 64, 32)

# Your code here

In [ ]:
#@title Click to reveal solution

architectures = [(8,), (32, 16), (64, 32, 16), (128, 64, 32)]

print(f"{'Architecture':20s} {'Train Acc':>10s} {'Test Acc':>10s} {'Parameters':>12s}")
print("-" * 55)

for arch in architectures:
    mlp_exp = MLPClassifier(
        hidden_layer_sizes=arch,
        activation='relu',
        max_iter=1000,
        random_state=42
    )
    mlp_exp.fit(X_train_scaled, y_train)

    train_acc = mlp_exp.score(X_train_scaled, y_train)
    test_acc = mlp_exp.score(X_test_scaled, y_test)

    # Count parameters
    n_params = sum(w.size for w in mlp_exp.coefs_) + sum(b.size for b in mlp_exp.intercepts_)

    print(f"{str(arch):20s} {train_acc:10.4f} {test_acc:10.4f} {n_params:12,}")

print()
print("Notice: bigger networks don't always generalize better.")
print("This is the bias-variance tradeoff again!")

---
# Summary

**Key takeaways from this notebook:**

- A perceptron with no hidden layers is equivalent to logistic regression (same weights, same boundary)
- Logistic regression fails on nonlinear problems like XOR; neural networks with hidden layers solve them by learning new feature representations
- ReLU is the default activation for hidden layers; sigmoid/tanh saturate and cause vanishing gradients
- Without nonlinear activations, stacking layers collapses to a single linear transformation
- Forward propagation is just repeated matrix multiplication + activation, layer by layer
- Every call to `.predict()` runs the forward pass we implemented from scratch

**What we haven't covered yet (Session 14):**
- How the weights are *learned* (backpropagation)
- Gradient descent variants (SGD, momentum, Adam)
- The full training loop
- Overfitting and regularization in neural networks
- PyTorch and Keras